# Задание 3 (агентный NLP)

Дедлайн:  17 мая, 23:59


В этом домашнем задании вам предстоит реализовать два шаблона агентов из лекции **без** использования LangChain, LangGraph или smolagents:

Предлагается повторить подходы из двух статей которые обсуждались на лекции:
1. **ReAct** (Yao et al., NeurIPS 2022) —  цикл агента *Мысль → Действие → Наблюдение* который прогоняется до тех пор, пока не будет получен окончательный ответ.
2. **Reflexion** (Shinn et al., NeurIPS 2023) — легковесная обертка для self-improvement с помощью текстовой рефлексии


Вам нужно оценить оба подхода на небольшом тестовом наборе из 10 multi-step вопросов и написать краткий анализ полученных результатов.

Для выполнения этого задания вам понадобится валидационный сет для оценки работы модели, `eval_set.json`, лежит в папке `homeworks`

## Ваша задача:
имплементировать ReAct цикл (`run_react`), обёртку Reflexion (`run_reflexion`) и скрипт оценки на валидационном сете.


Внимание! Полные баллы за задачу будут выставлены только при наличии анализа и обсуждения полученных результатов.

In [ ]:
!pip install -q mistralai wikipedia-api

In [ ]:
import os
import re
import json
import time
from typing import Optional

from mistralai.client import Mistral
import wikipediaapi

MISTRAL_API_KEY = os.environ.get('MISTRAL_API_KEY', 'PUT_YOUR_KEY_HERE')
MODEL = 'mistral-small-latest'

client = Mistral(api_key=MISTRAL_API_KEY)

def llm_call(messages, temperature=0.0, max_tokens=512):
    resp = client.chat.complete(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content

print(llm_call([{'role': 'user', 'content': 'Say hi in one word.'}]))

## 1. Инструменты для работы с агентом

- `wikipedia_search(query)` — возвращает первые 5 предложений суммаризации статьи из Википедии.
- `calculator(expression)` — запускает подсчет математического выражения внутри Python

In [ ]:
_wiki = wikipediaapi.Wikipedia(
    user_agent='MIPT-NLP-HW11/1.0',
    language='en',
)

def wikipedia_search(query: str, context: int = 1500) -> str:
    page = _wiki.page(query)
    if not page.exists():
        return f'ERROR: Wikipedia page "{query}" not found.'
    summary = page.summary
    sentences = summary.split('. ')[:5]
    text = '. '.join(sentences)
    if not text.endswith('.'):
        text += '.'
    return text[:context]

def calculator(expression: str) -> str:
    """Supports +, -, *, /, **, %, parentheses, and math functions."""
    import math
    safe_globals = {'__builtins__': {}, 'math': math, 'sqrt': math.sqrt,
                    'log': math.log, 'sin': math.sin, 'cos': math.cos, 'pi': math.pi,
                    'round': round, 'abs': abs, 'min': min, 'max': max}
    try:
        result = eval(expression, safe_globals, {})
        return str(result)
    except Exception as e:
        return f'ERROR: {type(e).__name__}: {e}'

TOOLS = {
    'wikipedia_search': wikipedia_search,
    'calculator':       calculator,
}


print(wikipedia_search('Albert Einstein')[:200])
print(calculator('2 ** 10 + sqrt(16)'))

## 2. ReAct инструменты

Стандартный ReAct блок:

```
Thought: I need to find Einstein's Nobel year.
Action: wikipedia_search
Action Input: Albert Einstein
```

После вызова можно добавить дополнительный блок:

```
Observation: <tool result>
```

... и вызвать модель заново. Цикл заканчивается когда модель печатает вот это:

```
Final Answer: <answer>
```

In [ ]:
REACT_SYSTEM_PROMPT = """You are a problem-solving agent that uses the ReAct format.

You have access to the following tools:
- wikipedia_search(query: str) — returns the first sentences of the English Wikipedia summary for the given query.
- calculator(expression: str) — evaluates a Python arithmetic expression. Supports +, -, *, /, **, %, sqrt(), log(), pi, round(), abs().

At each step, output EXACTLY ONE of the following two block formats and then STOP. Do not output anything after the block.

FORMAT A (use a tool):
Thought: <your reasoning about what to do next>
Action: <one of: wikipedia_search, calculator>
Action Input: <the argument to the tool, on a single line>

FORMAT B (you have the answer):
Thought: <your reasoning>
Final Answer: <the final answer, as concise as possible — usually a single number or short phrase>

Rules:
- Use tools when you need facts you don't know or arithmetic you can't reliably do in your head.
- Action Input must be on a single line, no quotes, no extra formatting.
- The Final Answer should be just the answer itself (a number, name, or short phrase) — not a sentence.
"""

In [ ]:
_FINAL_RE  = re.compile(r'Final Answer\s*:\s*(.+)', re.IGNORECASE | re.DOTALL)
_ACTION_RE = re.compile(r'Action\s*:\s*([a-zA-Z_]+)', re.IGNORECASE)
_INPUT_RE  = re.compile(r'Action Input\s*:\s*(.+?)(?:\n|$)', re.IGNORECASE | re.DOTALL)

def parse_react_step(text: str) -> dict:
    """Parse one ReAct step from raw LLM output.

    Returns a dict with one of:
      {'kind': 'final',  'answer': str}
      {'kind': 'action', 'tool': str, 'input': str}
      {'kind': 'error',  'reason': str, 'raw': str}
    """
    m_final = _FINAL_RE.search(text)
    if m_final:
        answer = m_final.group(1).strip()
        #
        answer = answer.split('\n')[0].strip()
        return {'kind': 'final', 'answer': answer}

    m_action = _ACTION_RE.search(text)
    m_input  = _INPUT_RE.search(text)
    if m_action and m_input:
        tool  = m_action.group(1).strip()
        inp   = m_input.group(1).strip().strip('"\'`')
        return {'kind': 'action', 'tool': tool, 'input': inp}

    return {'kind': 'error', 'reason': 'could not parse Action/Final Answer', 'raw': text}


Протестируем что все корректно отрабатывает:

In [ ]:
_ex1 = '''Thought: I need to look this up.
Action: wikipedia_search
Action Input: Marie Curie'''
_ex2 = '''Thought: Done.
Final Answer: 1480'''

print(parse_react_step(_ex1))
print(parse_react_step(_ex2))
print(parse_react_step('garbage output'))

## Часть 1 - Имплементировать ReAct цикл (7 баллов)


Вам нужно чередовать вызовы LLM с выполнением запрошенного инструмента или возвратом Final Answer. Диалог расширяется ответами ассистента + сообщениями пользователя вида `Observation: ...`, добавляемых после каждого вызова инструмента. 

Условия остановки: получение Final Answer, достижение `max_steps` вызовов LLM либо иное обоснованное условие. Реализация должна корректно обрабатывать неизвестные имена инструментов и ошибки парсинга без аварийного завершения.


Фиксируйте шаги выполнения в словаре trace. Он будет использован при анализе результатов, а также передаваться компоненту Reflexion.


*Указания:* подумайте как парсер может сообщить о некорректном выводе модели и добавьте это в вашу функцию.

In [ ]:
def run_react(question: str,
              max_steps: int = 6,
              verbose: bool = False) -> dict:
    """Run a ReAct loop on `question`.

    Args:
        question: the user's question.
        max_steps: hard cap on LLM calls.
        verbose: if True, print each step.

    Returns dict with keys:
        'answer':  the final answer string, or None if not finished.
        'steps':   number of LLM calls used.
        'trace':   list of step dicts (your design — log enough for analysis).
        'success': True iff a Final Answer was produced
    """
    system = REACT_SYSTEM_PROMPT

    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': f'Question: {question}'},
    ]
    trace = []

    # --- TODO: implement the loop ---
    raise NotImplementedError('TODO: implement run_react')

    # return {'answer': ..., 'steps': ..., 'trace': trace, 'success': ...}

Проверьте что все работает

In [ ]:
result = run_react('In what year did the first human walk on the Moon? How many years ago was that, counting from the year 2025?',
                   verbose=True)
print('ANSWER:', result['answer'])
print('STEPS:', result['steps'])

## Часть 2 - Цикл самоулучшения Reflexion (5 баллов)


Запустите ReAct; при неудаче запросите у LLM краткую рефлексию (а именно что пошло не так и какую стратегию применить далее), передав ей trace попытки который вы сохранили ранее, и повторите ReAct с полученными рефлексиями в `extra_system=`.

Верните лучший полученный ответ, количество шагов, все полученный рефлексии и также снова trace.


In [ ]:
REFLECTION_PROMPT = """You attempted to solve a question using a ReAct agent and the attempt failed.
Here is the question and the trace of what the agent did:

Question: {question}

Trace:
{trace}

Write a SHORT reflection (2–4 sentences) on what went wrong and what specific strategy the agent should try differently next time. Do not solve the question yourself — only reflect on the strategy.
"""

Задайте функцию форматирования на основе структуры, которую вы задали в run_react для trace-словарей.

In [ ]:


def format_trace_for_reflection(trace: list) -> str:
    # TODO
    raise NotImplementedError('TODO: format_trace_for_reflection')


In [ ]:

from ast import Pass


def run_reflexion(question: str,
                  max_attempts: int = 3,
                  max_steps_per_attempt: int = 6,
                  verbose: bool = False) -> dict:
    """Run ReAct with Reflexion retries.

    Returns dict with keys:
        'answer':       final answer string or None.
        'attempts':     number of ReAct attempts used (1..max_attempts).
        'total_steps':  sum of steps across all attempts.
        'reflections':  list of reflection strings (one per failed attempt).
        'traces':       list of traces (one per attempt).
    """
    # --- TODO: implement ---
    reflections = []
    traces = []
    for attempt in range(max_attempts):
        pass
    raise NotImplementedError('TODO: implement run_reflexion')

Функция оценки:

загружает `eval_set.json`, запускает обоих агентов, оценивает их работу и собирает все в таблицу

In [ ]:
with open('eval_set.json') as f:
    EVAL = json.load(f)

QUESTIONS = EVAL['questions']
print(f'Loaded {len(QUESTIONS)} questions.')
print(QUESTIONS)

In [ ]:
def grade(answer: Optional[str], q: dict) -> bool:
    """Lenient grader: case-insensitive substring match against any alias."""
    if answer is None:
        return False
    a = answer.strip().lower().rstrip('.').replace(',', '')
    for alias in q['answer_aliases']:
        if alias.lower().replace(',', '') in a:
            return True
    return False

def evaluate(agent_fn, name: str, sleep_between: float = 1.0):
    """Run agent_fn(question_str) -> result_dict on every eval question.
    The result_dict must have an 'answer' key. Other keys (steps/total_steps) are logged if present.
    """
    rows = []
    for q in QUESTIONS:
        print(f'[{name}] Q{q["id"]}: {q["question"][:60]}...')
        try:
            r = agent_fn(q['question'])
            ans   = r.get('answer')
            steps = r.get('total_steps', r.get('steps'))
        except Exception as e:
            print(f'  EXCEPTION: {e}')
            ans, steps = None, None
            r = {'error': str(e)}
        correct = grade(ans, q)
        rows.append({
            'id':       q['id'],
            'question': q['question'][:50] + '...',
            'answer':   ans,
            'expected': q['answer_aliases'][0],
            'correct':  correct,
            'steps':    steps,
            'trace':    r.get('trace'),     # ReAct
            'traces':   r.get('traces'),    # Reflexion
        })
        print(f'  -> {ans!r}  ({"OK" if correct else "WRONG"})')
        time.sleep(sleep_between)
    return rows

def print_summary(name: str, rows: list):
    n = len(rows)
    correct = sum(1 for r in rows if r['correct'])
    avg_steps = sum((r['steps'] or 0) for r in rows) / n
    print(f'\n=== {name} ===')
    print(f'Accuracy:  {correct}/{n} = {correct/n:.0%}')
    print(f'Avg steps: {avg_steps:.1f}')


##  Часть 3 — Проведение экспериментов (1 балл)


Запустите обоих агентов на полном валидационном сете и сохраните результаты:

Это потребует примерно  ~60–120 LLM запросов. У Mistral есть ограничения по количеству запросов в единицу времени, поэтому наша функция оценки ходит с интервалом в 1 секунду, поэтому выполнение команды может занимать от 5 до 15 минут

In [ ]:
react_rows = evaluate(lambda q: run_react(q, max_steps=6), name='ReAct')
print_summary('ReAct', react_rows)

In [ ]:
reflexion_rows = evaluate(lambda q: run_reflexion(q, max_attempts=3, max_steps_per_attempt=6), name='Reflexion')
print_summary('Reflexion', reflexion_rows)

In [ ]:
print(f'{"ID":<3} {"ReAct":<8} {"Reflexion":<10}  Question')
print('-' * 80)
for rr, rf in zip(react_rows, reflexion_rows):
    rr_mark = '✓' if rr['correct'] else '✗'
    rf_mark = '✓' if rf['correct'] else '✗'
    delta = ''
    if not rr['correct'] and rf['correct']:
        delta = '  <-- Reflexion fixed'
    elif rr['correct'] and not rf['correct']:
        delta = '  <-- Reflexion broke'
    print(f'{rr["id"]:<3} {rr_mark:<8} {rf_mark:<10}  {rr["question"]}{delta}')

## Часть 4 — Анализ trace-словарей (3 балла)

Имея результаты ReAct и Reflexion, подсчитайте статистики, которые помогут вам в финальном анализе. Реализуйте две функции:

- `analyze_traces(rows)` — принимает список результатов одного прогона (`react_rows` или `reflexion_rows`) и возвращает словарь с агрегированными метриками: распределение типов шагов, средняя длина трассы для правильных vs неправильных ответов, частота вызова каждого инструмента.
- `compare_runs(rows_a, rows_b, name_a, name_b)` — печатает сравнительную таблицу по двум прогонам: на каких вопросах один подход выиграл, а другой проиграл; средний расход шагов; общие метрики из `analyze_traces`.

Используйте результаты из предыдущей части (Анализ): вместо общих утверждений ссылайтесь на конкретные числа из этих функций.

*Подсказка:* в `react_rows` каждая строка содержит только финальный ответ и число шагов, но не сам trace. Чтобы анализировать содержимое трасс (типы шагов, какие инструменты вызывались), модифицируйте функцию `evaluate` так, чтобы она дополнительно сохраняла `trace` (для ReAct) или `traces` (для Reflexion) в каждую строку результата.

In [ ]:
from collections import Counter

def avg(xs):
    return sum(xs) / len(xs) if xs else 0.0


def analyze_traces(rows: list) -> dict:
    """Aggregate statistics over a list of result rows.
    
    Each row must contain: 'correct' (bool), 'steps' (int), and either
    'trace' (list of step dicts, for ReAct) or 'traces' (list of traces, for Reflexion).
    """
    step_kinds  = Counter()
    tool_calls  = Counter()
    steps_correct, steps_wrong = [], []
    parse_error_rate = 0
    accuracy = 0
    
    for r in rows:
        pass

    return {
        'n_questions': len(rows),
        'accuracy': accuracy,
        'avg_steps_correct': avg(steps_correct),
        'avg_steps_wrong': avg(steps_wrong),
        'step_kinds':  dict(step_kinds),
        'tool_calls': tool_calls,        
        'parse_error_rate': parse_error_rate,
    }

In [ ]:
def compare_runs(rows_a: list, rows_b: list, name_a: str = 'A', name_b: str = 'B'):
    """Print a side-by-side comparison of two evaluation runs."""
    stats_a = analyze_traces(rows_a)
    stats_b = analyze_traces(rows_b)
 
    a_only_correct =  0 #TODO
    b_only_correct =  0 #TODO
    both_correct   =  0 #TODO
    both_wrong     =  0 #TODO
    
    print(f'\n=== {name_a}  vs  {name_b} ===')
    print(f'Accuracy:           {stats_a["accuracy"]:.0%}    {stats_b["accuracy"]:.0%}')
    print(f'Avg steps (right):  {stats_a["avg_steps_correct"]:.1f}    {stats_b["avg_steps_correct"]:.1f}')
    print(f'Avg steps (wrong):  {stats_a["avg_steps_wrong"]:.1f}    {stats_b["avg_steps_wrong"]:.1f}')
    print(f'Parse error rate:   {stats_a["parse_error_rate"]:.1%}    {stats_b["parse_error_rate"]:.1%}')
    print(f'\nTool usage:')
    all_tools = set(stats_a['tool_calls']) | set(stats_b['tool_calls'])
    for t in sorted(all_tools):
        print(f'  {t:20s}  {stats_a["tool_calls"].get(t, 0):4d}  {stats_b["tool_calls"].get(t, 0):4d}')
    print(f'\nBoth correct:       {both_correct}')
    print(f'Both wrong:         {both_wrong}')
    print(f'Only {name_a}:           {a_only_correct}')
    print(f'Only {name_b}:           {b_only_correct}')

## Часть 5 — Анализ результатов (4 балла)

Проведите анализ полученных результатов опираясь на вывод `compare_runs` и содержимое trace-словарей, а именно опишите

1. Точнсть обоих агентов, а также среднее число шагов которые потребовались каждому из них для выполнения запроса. На какие вопросы Reflexion повлиял (помог дать верный ответ / повлиял на то что итоговый ответ неверный / никак не повлиял)? Используйте конкретные числа из `compare_runs`,
2. Выберите 2-3 вопрос на которые ReAct дал неверный ответ. Для каждого из них проанализируйте trace. В чем потенциально была проблема? Классифицируйте ошибку: сбой инструмента,  ошибка рассуждения или ошибка формата.
3. Для этих вопросов проверьте, помог ли агент Reflexion выполнить запрос корректно? Оцените рефлексию, которую предоставил агент на то, была ли она полезной или нет. Было ли такое, что агент рефлексии наоборот сделал так, что на вопрос на который  ReAct агент изначально ответил правильно, после рефлексии ReAct выдал некорректный ответ.
4. В конце оцените, есть ли вообще смысл подключать Reflexion или нет на таком бенчмарке. Что говорит разница `avg_steps_correct` vs `avg_steps_wrong` о поведении агентов?